# Actividad Evaluativa 1
##### integrantes: victor manuel hernandez fernandez
##### programa: Ingeniería de sistemas 

### 1. Preparar el entorno e importar la clase de carga
El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar src/clases_po 
y se agrega src a sys.path. Así se puede importar HFDatasetLoader sin instalar el proyecto como paquete.


In [54]:
import sys

print("Versión de Python:", sys.version)
print("Ubicación de Python:", sys.executable)

Versión de Python: 3.14.2 (main, Sep 10 2026, 12:33:04) [GCC 13.3.0]
Ubicación de Python: /workspaces/MachineLearning/.venv/bin/python


In [55]:

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

In [56]:
# 2. Configurar el dataset y la salida

In [57]:
hfdataset = HFDatasetLoader(
    dataset_name="smart-dashcam/motorcycle-accident-driving-datasets",
    split="train",
    filename="datos/Motorcycle-Accident-Driving-Datasets.csv",
)

In [58]:
# 3. Descargar el dataset y crear el DataFrame

In [59]:
df = hfdataset.load()
df.head(2)

,filename,case,duration
0,accident_000.mp4,accident,8.31
1,accident_001.mp4,accident,5.35


In [60]:
# 4. Revisar la estructura del DataFrame

In [61]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

Filas: 136
Columnas: 3
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 136 entries, 0 to 135
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   filename  136 non-null    object 
 1   case      136 non-null    object 
 2   duration  136 non-null    float32
dtypes: float32(1), object(2)
memory usage: 2.8+ KB


### 5. Transformación Mínima de los Datos
##### se implementaron cuatro transformaciones consecutivas:

1. **Estandarización de identificadores:** Se renombraron las columnas (`filename` $\rightarrow$ `archivo`, `case` $\rightarrow$ `tipo_evento`, `duration` $\rightarrow$ `duracion_segundos`) para facilitar la interpretación del contexto en español.
2. **Filtrado de umbral temporal:** Se filtraron únicamente los videos con `duracion_segundos > 5.0`, descartando secuencias con información visual insuficiente.
3. **Ingeniería de variables (Feature Engineering):** Se construyó la característica booleana `es_video_largo` (donde valores $\ge 7.0$ segundos se etiquetan como `True`), facilitando futuros análisis de segmentación o clasificación.
4. **Formato numérico y reindexación:** Se normalizó la precisión decimal de `duracion_segundos` y se reestructuró el índice (`reset_index`) para mantener la integridad de los registros tabulares.


In [62]:
# 1
df_renombrado = df.rename(columns={
    "filename": "archivo",
    "case": "tipo_evento",
    "duration": "duracion_segundos"
})

# 2
df_filtrado = df_renombrado[df_renombrado["duracion_segundos"] > 5.0].reset_index(drop=True)
#3
df_transformado = df_filtrado.assign(
    duracion_segundos=df_filtrado["duracion_segundos"].round(2),
    es_video_largo=df_filtrado["duracion_segundos"] >= 7.0
)

# Verificación de la transformación
print("=== RESUMEN DE TRANSFORMACIONES APLICADAS ===")
print(f"Dimensiones originales:    {df.shape}")
print(f"Dimensiones transformadas: {df_transformado.shape}")
print("\nPrimeras 5 filas con las 4 transformaciones:")
df_transformado.head(5)

=== RESUMEN DE TRANSFORMACIONES APLICADAS ===
Dimensiones originales:    (136, 3)
Dimensiones transformadas: (91, 4)

Primeras 5 filas con las 4 transformaciones:


,archivo,tipo_evento,duracion_segundos,es_video_largo
0,accident_000.mp4,accident,8.31,True
1,accident_001.mp4,accident,5.35,False
2,accident_0010.mp4,accident,6.71,False
3,accident_0011.mp4,accident,6.40,False
4,accident_0012.mp4,accident,7.08,True


### 5. Importar la clase de reportes

In [63]:
from clases_po import Reportes

### 6. Generación del Reporte del DataFrame Transformado
Instanciamos la clase `Reportes` y le pasamos el DataFrame resultante (`df_transformado`) para documentar su estructura en formato de texto.

In [64]:
from clases_po import Reportes

reportes = Reportes()

# Generar y visualizar el reporte del DataFrame transformado
texto = reportes.generar_reporte(df_transformado)
print(texto)

REPORTE DEL DATAFRAME
Filas: 91
Columnas: 4
Duplicados: 0

Columnas y tipos:
- archivo: object
- tipo_evento: object
- duracion_segundos: float32
- es_video_largo: bool

Valores nulos por columna:
- archivo: 0
- tipo_evento: 0
- duracion_segundos: 0
- es_video_largo: 0

Estadísticas numéricas:
       duracion_segundos
count             91.000
mean               6.307
std                1.320
min                5.010
25%                5.720
50%                5.720
75%                6.440
max               13.150



### 7. guardar reporte transformado

In [65]:
# Guardar el archivo .txt con el resumen de los datos transformados
ruta = reportes.guardar_reporte(
    df_transformado,
    "reportes/motorcycle_accident_transformed_summary.txt",
)

print(f"Reporte del DataFrame transformado guardado con éxito en: {ruta}")

Reporte del DataFrame transformado guardado con éxito en: reportes/motorcycle_accident_transformed_summary.txt


# Reporte original

In [66]:
reportes = Reportes()
texto = reportes.generar_reporte(df)

print(texto)

REPORTE DEL DATAFRAME
Filas: 136
Columnas: 3
Duplicados: 0

Columnas y tipos:
- filename: object
- case: object
- duration: float32

Valores nulos por columna:
- filename: 0
- case: 0
- duration: 0

Estadísticas numéricas:
       duration
count   136.000
mean      5.417
std       1.754
min       1.470
25%       4.380
50%       5.720
75%       5.783
max      13.150



#### Imprimir el reporte directamente

In [67]:
reportes.imprimir_reporte(df)

REPORTE DEL DATAFRAME
Filas: 136
Columnas: 3
Duplicados: 0

Columnas y tipos:
- filename: object
- case: object
- duration: float32

Valores nulos por columna:
- filename: 0
- case: 0
- duration: 0

Estadísticas numéricas:
       duration
count   136.000
mean      5.417
std       1.754
min       1.470
25%       4.380
50%       5.720
75%       5.783
max      13.150


### 8. Guardar el reporte ORIGINAL en un archivo de texto

In [68]:
ruta = reportes.guardar_reporte(
    df,
    "reportes/motorcycle-accident-driving-datasets-ORIGINAL.txt",
)

print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes/motorcycle-accident-driving-datasets-ORIGINAL.txt


# CONCLUSIONES 

Tras la inspección inicial del conjunto de datos smart-dashcam/motorcycle-accident-driving-datasets, se constató la ausencia total de valores nulos o registros duplicados en sus 136 observaciones originales, lo que proporciona una base limpia.

Se hace unas transformaciones haciendo un renombrado de variables (archivo, tipo_evento, duracion_segundos) lo que facilita la legibilidad del código y la interpretación del contexto del problema en español.

Con la aplicación del filtro condicional (duracion_segundos > 5.0) permitió descartar secuencias excesivamente breves que podrían no contener suficiente informacion de los accidentes